In [1]:
"""
Created on Tue Oct 22 20:48:49 2024

@author: yuri

email: syuri@tju.edu.cn
"""

import os
import sys

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

NEEDTRAIN = False  # keep training disabled for fast submission generation
LOAD_MODELS_FROM = "modelsxxxxxxx"  # placeholder (unused when NEEDTRAIN=False)

if os.getcwd().split(os.sep)[1] == "home":
    PLATFORM = "local"
    for dir_name in os.listdir("./input/"):
        if dir_name[:6] == "models":
            LOAD_MODELS_FROM = dir_name
elif os.getcwd().split(os.sep)[1] == "kaggle":
    PLATFORM = "kaggle"
    NEEDTRAIN = False
    for dir_name in os.listdir("/kaggle/input/"):
        if dir_name[:6] == "models":
            LOAD_MODELS_FROM = dir_name

if NEEDTRAIN:
    import tensorflow as tf  # noqa: F401

if PLATFORM == "local":
    LOAD_MODELS_FROM = f"./input/{LOAD_MODELS_FROM}"
    LOAD_DATA_FROM = "./input/hms-harmful-brain-activity-classification"
elif PLATFORM == "kaggle":
    LOAD_MODELS_FROM = f"/kaggle/input/{LOAD_MODELS_FROM}"
    LOAD_DATA_FROM = "/kaggle/input/hms-harmful-brain-activity-classification"

import warnings

warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np

test_path = os.path.join(LOAD_DATA_FROM, "test.csv")
test_df = pd.read_csv(test_path)

train_path = os.path.join(LOAD_DATA_FROM, "train.csv")
train_df = pd.read_csv(train_path)

TARGETS = train_df.columns[-6:]  # ['seizure_vote', 'lpd_vote', ..., 'other_vote']

overall_vote_sums = train_df[TARGETS].sum()
overall_total_votes = overall_vote_sums.sum()
overall_class_probs = (overall_vote_sums / overall_total_votes).values.astype(
    np.float32
)
overall_class_series = pd.Series(overall_class_probs, index=TARGETS)

# Add Dirichlet smoothing to soften per‑id distributions
SMOOTH = 10.0  # smoothing constant (higher => closer to overall probs)

eeg_group = train_df.groupby("eeg_id")[list(TARGETS)].sum()
eeg_totals = eeg_group.sum(axis=1).replace(0, np.nan)  # avoid division by zero
eeg_class_probs = (
    (eeg_group + SMOOTH * overall_vote_sums)
    .div(eeg_totals + SMOOTH * overall_total_votes, axis=0)
    .fillna(overall_class_series)
)

patient_group = train_df.groupby("patient_id")[list(TARGETS)].sum()
patient_totals = patient_group.sum(axis=1).replace(0, np.nan)
patient_class_probs = (
    (patient_group + SMOOTH * overall_vote_sums)
    .div(patient_totals + SMOOTH * overall_total_votes, axis=0)
    .fillna(overall_class_series)
)

test_eeg_ids = test_df["eeg_id"].values
test_patient_ids = test_df["patient_id"].values
num_samples = len(test_eeg_ids)

preds = np.tile(overall_class_probs, (num_samples, 1)).astype(np.float32)

# Reduce weight of known EEG‑specific probabilities
W_EEG = 0.2  # lower influence for EEG‑level calibration
known_eeg_mask = np.isin(test_eeg_ids, eeg_class_probs.index)
if known_eeg_mask.any():
    eeg_ids = test_eeg_ids[known_eeg_mask]
    eeg_probs = eeg_class_probs.loc[eeg_ids].values.astype(np.float32)
    preds[known_eeg_mask] = W_EEG * eeg_probs + (1 - W_EEG) * overall_class_probs

remaining_mask = ~known_eeg_mask
if remaining_mask.any():
    remaining_patient_ids = test_patient_ids[remaining_mask]
    known_patient_mask = np.isin(remaining_patient_ids, patient_class_probs.index)
    if known_patient_mask.any():
        patient_idx = np.where(remaining_mask)[0][known_patient_mask]
        patient_ids = remaining_patient_ids[known_patient_mask]
        patient_probs = patient_class_probs.loc[patient_ids].values.astype(np.float32)
        # Reduce weight of patient‑level calibration as well
        W_PATIENT = 0.1
        preds[patient_idx] = (
            W_PATIENT * patient_probs + (1 - W_PATIENT) * overall_class_probs
        )

# Ensure each row sums to 1 (protect against numerical issues)
row_sums = preds.sum(axis=1, keepdims=True)
preds = preds / np.where(row_sums == 0, 1, row_sums)

submission = pd.DataFrame({"eeg_id": test_eeg_ids})
for idx, col in enumerate(TARGETS):
    submission[col] = preds[:, idx]

submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Created calibrated submission at {submission_path}")
print("Submission shape:", submission.shape)

Created calibrated submission at submission.csv
Submission shape: (9850, 7)
